# SQL, and where the data actually lives

MichAl Academy, lesson 1.5.

Run each cell with **Shift+Enter**. `sqlite3` ships with Python, so there is
nothing to install and no server to connect to. The database lives in memory and
disappears when the runtime stops.

Same eight hosts as lesson 1.4, so you can compare every answer against the
pandas version you already know.

## 1. Make a database

In [ ]:
import sqlite3
import pandas as pd
from io import StringIO

CSV = '''host,tld,length,age_days,flagged
login.acme.co,co,13,1240,False
cdn.acme.co,co,11,980,False
acme-secure.tk,tk,14,3,True
mail.acme.co,co,12,1100,True
acme-login.tk,tk,13,,True
docs.acme.co,co,12,640,False
acme.verify.tk,tk,14,1,True
api.acme.co,co,11,1500,False
'''

df = pd.read_csv(StringIO(CSV))

owners = pd.DataFrame({
    "host":  ["login.acme.co", "cdn.acme.co", "mail.acme.co"],
    "owner": ["it", "web", "it"],
})

con = sqlite3.connect(":memory:")
df.to_sql("hosts", con, index=False)
owners.to_sql("owners", con, index=False)

def q(sql):
    """Run a query and hand the result back as a DataFrame."""
    return pd.read_sql_query(sql, con)

print("tables:", q("SELECT name FROM sqlite_master WHERE type='table'")["name"].tolist())

`pd.read_sql_query` is the bridge. A query goes in, a DataFrame comes out, and
from that point everything from lesson 1.4 applies.

## 2. The same four operations

### Filter rows

In [ ]:
sql_result = q("SELECT host, tld, flagged FROM hosts WHERE tld = 'tk'")
pd_result  = df.loc[df["tld"] == "tk", ["host", "tld", "flagged"]]

print(sql_result)
print()
print("same number of rows?", len(sql_result) == len(pd_result))
print("flagged came back as:", sql_result["flagged"].tolist())

Three rows either way. Note the `flagged` values: SQLite has no boolean type, so
`True` is stored and returned as `1`. Types are looser in SQL than in a
DataFrame, so check what you actually got, the same habit as checking `dtypes`.

### Pick columns

In [ ]:
print(q("SELECT tld, flagged FROM hosts").shape, "  <- SQL")
print(df[["tld", "flagged"]].shape, "  <- pandas")

### Group

In [ ]:
sql_group = q('''
    SELECT tld,
           COUNT(*)     AS hosts,
           AVG(flagged) AS flagged_rate
    FROM hosts
    GROUP BY tld
''')

pd_group = df.groupby("tld").agg(
    hosts=("flagged", "size"),
    flagged_rate=("flagged", "mean"),
).reset_index()

print(sql_group.to_string(index=False))
print()
print(pd_group.to_string(index=False))

The same two numbers lesson 1.4 arrived at. `GROUP BY` plus an aggregate in one
statement is `groupby` plus `agg` in two calls.

### Join

In [ ]:
sql_join = q('''
    SELECT h.host, o.owner
    FROM hosts h
    LEFT JOIN owners o ON h.host = o.host
''')

pd_join = df.merge(owners, on="host", how="left")[["host", "owner"]]

print(sql_join.to_string(index=False))
print()
print("rows: SQL", len(sql_join), " pandas", len(pd_join))

`LEFT JOIN` keeps every row on the left and puts NULL where the right side had no
match. That is `how="left"`, down to the blanks.

And the same warning as last lesson: check the row count before and after. A join
that grows the table means the key was not unique on the other side.

## 3. Read it in the order it runs

SQL is written in one order and evaluated in roughly another.

```
FROM      which table
JOIN      what else to bring alongside
WHERE     which rows to keep
GROUP BY  what to collapse them into
HAVING    which of those groups to keep
SELECT    which columns to show
ORDER BY  how to sort what is left
LIMIT     how much of it to return
```

`WHERE` filters rows **before** grouping. `HAVING` filters groups **after**.
Getting those two the wrong way round is the most common SQL mistake there is.

In [ ]:
# WHERE filters rows first, so this counts only the .tk rows
print(q("SELECT tld, COUNT(*) n FROM hosts WHERE tld = 'tk' GROUP BY tld").to_string(index=False))
print()
# HAVING filters the groups afterwards, so this counts everything then drops small groups
print(q("SELECT tld, COUNT(*) n FROM hosts GROUP BY tld HAVING COUNT(*) > 3").to_string(index=False))

## 4. NULL is NaN with sharper edges

`acme-login.tk` still has no `age_days`. Two of its behaviours match pandas and
one is worse.

In [ ]:
# Same as pandas: a comparison against NULL is not true, so the row is missed
young = q("SELECT host, age_days, flagged FROM hosts WHERE age_days < 30")

print(young.to_string(index=False))
print()
print("rows returned:", len(young), " (the flagged .tk domain with no age is not here)")

In [ ]:
# Same as pandas: aggregates skip NULL rather than poisoning the answer
print(q('''
    SELECT AVG(age_days)   AS mean,
           COUNT(age_days) AS non_null,
           COUNT(*)        AS rows
    FROM hosts
''').to_string(index=False))

`COUNT(*)` counts rows. `COUNT(column)` counts non-NULL values in that column.
When those two numbers differ, you have found your missing data.

Now the one that is SQL's alone.

In [ ]:
print("= NULL  ->", q("SELECT host FROM hosts WHERE age_days = NULL")["host"].tolist())
print("IS NULL ->", q("SELECT host FROM hosts WHERE age_days IS NULL")["host"].tolist())

`= NULL` is not false, it is **unknown**, and a `WHERE` clause keeps only rows
that are true. Unknown gets discarded. The query runs, reports no error, and
returns an empty result that reads like a clean bill of health.

Use `IS NULL` and `IS NOT NULL`. Always.

Sorting has its own surprise.

In [ ]:
print(q("SELECT host, age_days FROM hosts ORDER BY age_days LIMIT 3").to_string(index=False))

SQLite treats NULL as smaller than any other value for sorting, so "show me the
newest domains" hands you the ones whose age nobody knows.

## 5. Never paste values into a query

This is the habit that matters most outside the notebook.

In [ ]:
tld = "tk"

# Wrong. If `tld` came from anywhere you do not control, this is an injection.
bad = f"SELECT COUNT(*) FROM hosts WHERE tld = '{tld}'"
print("built by hand:", bad)

# Right. The value travels separately and is never parsed as SQL.
cur = con.execute("SELECT COUNT(*) FROM hosts WHERE tld = ?", (tld,))
print("parameterised result:", cur.fetchone()[0])

In [ ]:
# What the difference buys you
attack = "tk' OR '1'='1"

hand_built = f"SELECT COUNT(*) FROM hosts WHERE tld = '{attack}'"
print("hand-built returns:  ", con.execute(hand_built).fetchone()[0], "rows")

safe = con.execute("SELECT COUNT(*) FROM hosts WHERE tld = ?", (attack,))
print("parameterised returns:", safe.fetchone()[0], "rows")

The hand-built version matched every row in the table. The parameterised version
looked for a top level domain literally called `tk' OR '1'='1`, found none, and
returned zero. The value was data, not code.

You already knew this. It is here because the first version is what everyone
writes in a notebook when they are in a hurry, and notebooks get promoted to
scripts.

## 6. Your turn

You want the hosts whose age is unknown, so somebody can go and find out. This
query returns nothing, and there is definitely one such host.

Change one thing so it finds it.

In [ ]:
unknown_age = q("SELECT host, tld, flagged FROM hosts WHERE age_days = NULL")

print(unknown_age.to_string(index=False))
print()
print("rows found:", len(unknown_age))
print("found the missing host?", len(unknown_age) == 1)

The query is valid SQL and the database is not lying to you. Ask what `age_days
= NULL` actually evaluates to.

<details>
<summary>Answer</summary>

Nothing equals NULL, not even NULL, because NULL means unknown and the answer to
"is this unknown value equal to that unknown value" is itself unknown. `WHERE`
keeps only rows that evaluate to true.

```sql
SELECT host, tld, flagged FROM hosts WHERE age_days IS NULL
```

The same applies in reverse: use `IS NOT NULL`, never `!= NULL`.

</details>

## What you now have

- Four operations you already knew, in a second language: `SELECT`, `WHERE`, `GROUP BY`, `JOIN`
- `WHERE` filters rows before grouping, `HAVING` filters groups after
- `COUNT(*)` against `COUNT(column)` finds your missing data in one line
- `IS NULL`, never `= NULL`, and NULLs sort first
- `?` placeholders, never string formatting
- Filter and aggregate in SQL, analyse in pandas. The SIEM is not going to fit in memory.

Next is lesson 1.6, plotting, and how to look at a chart without being fooled by it.